# Clase 1 — Anatomía de un prompt

Un **prompt** es la instrucción que le das a un modelo de lenguaje. La diferencia entre una respuesta útil y una vaga casi siempre está en cómo escribiste esa instrucción — no en el modelo.

En esta clase vamos a descomponer un prompt en sus partes, entender por qué cada una importa, y comparar en tiempo real qué pasa cuando las usás bien o mal.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Las cinco partes de un prompt |
| 3 | Armar el wrapper LLM |
| 4 | Experimento: prompt vacío vs. prompt estructurado |
| 5 | Ajustar cada parte por separado |
| 6 | Actividad práctica |


Idea clave
----------
El notebook usa una sola función (`llamar_llm`) como interfaz común.
Así, el resto de las celdas no necesita cambiar aunque cambie el backend.
```

---
## 1. Configuración del entorno

Este notebook puede correr de dos formas:

| Backend | Qué necesitás | Cuándo usarlo |
|---|---|---|
| `local` | llama.cpp instalado + modelo GGUF descargado | Sin internet, máquina con ≥4 GB RAM |
| `gemini` | Cuenta Google, API key gratuita | Recomendado si es tu primera vez |

### Obtener tu API key de Gemini (solo una vez)

1. Entrá a [aistudio.google.com](https://aistudio.google.com) con tu cuenta Google.
2. Hacé clic en **Get API key** → **Create API key**.
3. Copiá la clave (empieza con `AIza...`).

### Guardar la key de forma segura

**Nunca** escribas la clave directamente en el código: si subís el notebook a GitHub, queda expuesta.
Lo correcto es guardarla en un archivo `.env` en la carpeta del proyecto:

```bash
# Ejecutá esto UNA VEZ en tu terminal, desde la carpeta del proyecto
echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
```

El archivo `.env` queda en tu máquina y **no** se sube al repositorio.
Si no querés crear el archivo, la celda siguiente te pide la clave de forma interactiva.

In [ ]:
# ─── Instalación de dependencias (solo la primera vez) ────────────────────────
# Descomentá la línea que necesites:
!pip install google-genai python-dotenv          
!pip install llama-cpp-python huggingface-hub  
!pip install ollama  
print("Listo para configurar.")

In [16]:
# ─── Elegí tu backend ─────────────────────────────────────────────────────────
# BACKEND = "gemini"   # Grato, pero requiere API key
BACKEND = "ollama"   # 🚀 RECOMENDADO: Con GPU automática. Ver instrucciones abajo
# BACKEND = "local"   # CPU solamente (sin GPU)

# Si usas "ollama", necesitas:
# 1. Descargar Ollama: https://ollama.ai
# 2. Instalar y ejecutar: ollama serve
# 3. En otra terminal: ollama pull llama2  (o tu modelo favorito)


In [18]:
import os
import getpass

# Modelos Gemini disponibles con tier gratuito:
GEMINI_MODEL = "gemini-2.0-flash-lite"   # rápido y gratuito gemini-2.0-flash-lite - gemini-2.5-flash
# GEMINI_MODEL = "gemma-4-26b-a4b-it"   # alternativa de Google DeepMind

# ─── Cargar API key ───────────────────────────────────────────────────────────
if BACKEND == "gemini":
    # Primero intenta leer del archivo .env
    try:
        from dotenv import load_dotenv
        load_dotenv()   # carga variables desde el archivo .env si existe
    except ImportError:
        pass  # si no está python-dotenv, sigue igual

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

    # Si no encontró la key en .env, la pide de forma interactiva
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini (no se muestra): ")

print(f"Backend seleccionado: {BACKEND}")
print("API key: OK" if BACKEND == "gemini" and GEMINI_API_KEY else "(modo local)")

Backend seleccionado: ollama
(modo local)


---
## 2. Las cinco partes de un prompt

Un prompt bien armado puede tener hasta cinco componentes. No todos son obligatorios en todas las situaciones, pero conocerlos te permite diagnosticar rápido por qué una respuesta salió mal.

| Componente | Qué hace | Ejemplo |
|---|---|---|
| **Rol** | Le dice al modelo *quién* debe ser | `"Sos un contador público con 10 años de experiencia"` |
| **Contexto** | Información de fondo relevante para la tarea | `"El cliente tiene una PyME de importación"` |
| **Instrucción** | La tarea concreta a realizar | `"Explicá las deducciones posibles en el impuesto a las ganancias"` |
| **Formato** | Cómo debe estructurarse la respuesta | `"En 3 puntos, sin tecnicismos"` |
| **Output esperado** | Qué debe incluir o evitar la respuesta | `"Incluí un ejemplo numérico simple al final"` |

_
> 💡 El componente que más se omite (y más falta hace) es el **formato**. Sin él, el modelo elige cómo estructurar la respuesta — y no siempre elige bien para tu caso de uso.

---
## 3. Armar el wrapper LLM

Para no repetir código en cada experimento, armamos una función única `llamar_llm()` que funciona con cualquiera de los dos backends. La firma es siempre la misma: preguntás y recibís texto.

In [19]:
# ─── Inicializar el cliente según el backend elegido ─────────────────────────

if BACKEND == "gemini":
    from google import genai
    from google.genai import types

    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
    print("✅ Cliente Gemini inicializado.")

elif BACKEND == "ollama":
    import ollama
    
    OLLAMA_MODEL = "gemma2:9b"  # ← Cambiar de llama2 a llama3.2
    
    print("🚀 Conectando a Ollama...")
    try:
        ollama.list()  # ← SIMPLIFICAR (quitar el código del bug)
        print(f"✅ Ollama disponible. Usando modelo: {OLLAMA_MODEL}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Solución: Abre otra terminal y ejecuta: ollama serve")
        raise

elif BACKEND == "local":
    import time
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    # Llama 3.2 3B — modelo mucho más potente
    REPO_ID = "bartowski/Llama-3.2-3B-Instruct-GGUF"
    FILENAME = "Llama-3.2-3B-Instruct-Q4_K_M.gguf"

    print("Descargando modelo local (puede tardar la primera vez)...")
    ruta_modelo = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
    
    # Configuración mejorada para GPU/CPU
    # n_gpu_layers: -1 = usar toda la GPU, 0 = solo CPU, 10-50 = mixto
    print("Inicializando modelo...")
    _llm_local = Llama(
        model_path=ruta_modelo, 
        n_ctx=8192,  # Contexto estándar (reducido de 16384 para mejor compatibilidad)
        n_gpu_layers=0,  # 0 = CPU solamente. Cámbialo a -1 si tienes CUDA compilado correctamente
        verbose=False  # Silencioso para menos ruido
    )
    print("✅ Modelo local listo (usando CPU).")

else:
    raise ValueError(f"Backend '{BACKEND}' no reconocido. Usá 'gemini', 'ollama' o 'local'.")

🚀 Conectando a Ollama...
✅ Ollama disponible. Usando modelo: gemma2:9b


In [20]:
def llamar_llm(
    prompt,
    system_prompt="follow the instructions. answer in spanish.",
    temperature=0.7,
    max_tokens=1000
):
    """Envía un prompt al modelo configurado y devuelve la respuesta como string."""

    if BACKEND == "gemini":
        respuesta = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return respuesta.text.strip()

    elif BACKEND == "ollama":
        respuesta = ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            system=system_prompt,
            stream=False,
            options={
                "temperature": temperature,
                "num_predict": max_tokens,
            }
        )
        return respuesta['response'].strip()

    elif BACKEND == "local":
        respuesta = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return respuesta["choices"][0]["message"]["content"].strip()


# Prueba rápida de conexión
print("Testing connection...")
resultado = llamar_llm("Decí 'Hola, estoy funcionando'", max_tokens=20)
print(f"✅ {resultado}")

Testing connection...
✅ Hola, estoy funcionando.


---
## 4. Experimento: prompt vacío vs. prompt estructurado

Vamos a hacer la misma consulta de tres formas distintas. La tarea en los tres casos es la misma: que el modelo explique qué es una red neuronal. Solo cambia cómo se la pedimos.

In [21]:
# ─── Versión 1: prompt mínimo — sin estructura ────────────────────────────────
prompt_minimo = "Explicá redes neuronales."

print("=" * 60)
print("VERSIÓN 1 — Prompt mínimo")
print("=" * 60)
print(llamar_llm(prompt_minimo))
print()

VERSIÓN 1 — Prompt mínimo
## Redes Neuronales: Un Vistazo

Las redes neuronales son sistemas informáticos inspirados en la estructura y función del cerebro humano.  

Imagina un grupo de neuronas interconectadas, cada una procesando información.  Estas "neuronas artificiales" se organizan en capas:

* **Capa de entrada:** Recibe los datos brutos, como imágenes o texto.
* **Capas ocultas:** Procesan la información mediante cálculos matemáticos complejos. Hay varias capas ocultas que permiten aprender patrones y relaciones complejas.
* **Capa de salida:** Produce la respuesta final, como una clasificación o predicción.

Las conexiones entre las neuronas tienen un "peso", que representa la importancia de esa conexión. Durante el entrenamiento, el algoritmo ajusta estos pesos para minimizar el error en las respuestas. 

**¿Cómo aprenden?**

A través del aprendizaje supervisado: se les alimenta con datos etiquetados (por ejemplo, imágenes con su etiqueta correspondiente). El algoritmo compa

In [22]:
# ─── Versión 2: con rol y formato, sin contexto ───────────────────────────────
prompt_con_rol = """Sos un profesor de tecnología.
Explicá qué es una red neuronal en exactamente 3 puntos breves"""

print("=" * 60)
print("VERSIÓN 2 — Con rol y formato")
print("=" * 60)
print(llamar_llm(prompt_con_rol))
print()

VERSIÓN 2 — Con rol y formato
Claro que sí, aquí te van las explicaciones en tres puntos sobre redes neuronales:

1. **Modelos inspirados en el cerebro:** Las redes neuronales son algoritmos que imitan la estructura y función del cerebro humano, con capas de "neuronas" interconectadas que procesan información. 
2. **Aprendizaje a través de datos:**  Las redes neuronales aprenden de grandes conjuntos de datos, ajustando los "pesos" de las conexiones entre neuronas para mejorar su capacidad de realizar una tarea específica, como reconocer imágenes o traducir idiomas.
3. **Aplicaciones diversas:** Las redes neuronales se utilizan en una amplia gama de aplicaciones, desde reconocimiento facial y conducción autónoma hasta diagnóstico médico y análisis de sentimientos.


Espero que esto te ayude a comprender mejor qué son las redes neuronales.



In [23]:
# ─── Versión 3: prompt completo con todos los componentes ─────────────────────
prompt_completo = """Rol: Sos un profesor universitario de inteligencia artificial.
Contexto: Estás explicando a estudiantes adultos que trabajan en empresas y no tienen
  experiencia previa en programación.
Instrucción: Explicá qué es una red neuronal artificial.
Formato: 3 bullets, máximo 2 líneas cada uno, sin fórmulas matemáticas.
Output esperado: Terminá con una analogía cotidiana que resuma la idea principal."""

print("=" * 60)
print("VERSIÓN 3 — Prompt completo")
print("=" * 60)
print(llamar_llm(prompt_completo))
print()

VERSIÓN 3 — Prompt completo
* Una red neuronal artificial es un sistema de software inspirado en el funcionamiento del cerebro humano.

* Está compuesta por capas de "neuronas" conectadas entre sí, cada una procesando información y transmitiéndola a las siguientes. 

* A través del entrenamiento con grandes cantidades de datos, estas conexiones se ajustan para que la red pueda realizar tareas como reconocimiento de imágenes o traducción de idiomas.
* Imaginen una orquesta donde cada músico (neurona) interpreta su parte individualmente, pero al sincronizar sus sonidos crean una melodía compleja y hermosa.



> 💡 **Para discutir:** ¿En qué versión la respuesta fue más útil para el público descrito? ¿Qué componente del prompt completo hizo más diferencia? ¿Hubo algún componente que parecía redundante?

---
## 5. Ajustar cada parte por separado

Ahora vamos a ver cómo cambia la respuesta cuando modificamos *un solo componente* a la vez. Esto entrena la intuición para saber dónde tocar cuando una respuesta no es la que esperabas.

In [24]:
# ─── Efecto del ROL ───────────────────────────────────────────────────────────
# La instrucción es idéntica, solo cambia quién se supone que responde.

instruccion_base = "Explicá en 2 oraciones por qué es importante proteger los datos personales."

roles = [
    "Sos un abogado especialista en privacidad.",
    "Sos un técnico de sistemas en una empresa mediana.",
    "Sos un periodista que escribe para el público general."
]

for rol in roles:
    print(f"Rol: {rol}")
    print("-" * 50)
    print(llamar_llm(instruccion_base, system_prompt=rol, max_tokens=120))
    print()

Rol: Sos un abogado especialista en privacidad.
--------------------------------------------------
Proteger los datos personales es crucial para preservar la autonomía e integridad de las personas, ya que les permite controlar cómo se utiliza su información y evitar el abuso o discriminación.  Además, la protección de datos fomenta la confianza en las relaciones digitales y el desarrollo de una sociedad más justa e inclusiva.

Rol: Sos un técnico de sistemas en una empresa mediana.
--------------------------------------------------
Proteger los datos personales es crucial para mantener la confianza de nuestros clientes y empleados.  Una violación de datos puede causar daños irreparables a nuestra reputación, finanzas y relaciones con las partes interesadas.

Rol: Sos un periodista que escribe para el público general.
--------------------------------------------------
Proteger nuestros datos personales es fundamental porque nos permite controlar quién tiene acceso a nuestra información 

In [25]:
# ─── Efecto del FORMATO ───────────────────────────────────────────────────────
# El contenido pedido es el mismo, pero el formato cambia completamente la usabilidad.

contenido = "Enumerá las ventajas de usar modelos de lenguaje en atención al cliente."

formatos = [
    "Respondé en prosa, como si fuera un párrafo de informe ejecutivo.",
    "Respondé en una lista de 4 ítems concisos, sin introducción.",
    "Respondé como una tabla con dos columnas: Ventaja | Por qué importa."
]

for fmt in formatos:
    print(f"Formato: {fmt}")
    print("-" * 50)
    print(llamar_llm(f"{contenido}\n{fmt}", max_tokens=180))
    print()

Formato: Respondé en prosa, como si fuera un párrafo de informe ejecutivo.
--------------------------------------------------
Los modelos de lenguaje presentan numerosas ventajas para la atención al cliente.  

Su capacidad para procesar grandes cantidades de información permite una respuesta rápida y precisa a consultas frecuentes, liberando tiempo valioso para agentes humanos que puedan abordar casos más complejos. Además, los modelos pueden ofrecer asistencia personalizada las 24 horas del día, los 7 días de la semana, mejorando la satisfacción del cliente y reduciendo tiempos de espera. Su capacidad de aprendizaje continuo permite una mejora constante en la calidad de las respuestas y una adaptación a las necesidades cambiantes de los clientes. Finalmente, la implementación de estos modelos puede reducir costos operativos al automatizar tareas repetitivas y liberar recursos humanos para funciones más estratégicas.

Formato: Respondé en una lista de 4 ítems concisos, sin introducció

---
## 6. Actividad práctica

Tomá la siguiente tarea y construí un prompt completo usando los cinco componentes. Después compará tu resultado con el prompt mínimo que se incluye abajo.

**Tarea:** pedirle al modelo que explique cómo funciona una tarjeta de crédito para alguien que nunca tuvo una usando el modelo local.

In [26]:
# ─── Prompt mínimo (punto de partida) ────────────────────────────────────────
prompt_minimo_actividad = "Explicá cómo funciona una tarjeta de crédito."

print("RESULTADO CON PROMPT MÍNIMO:")
print("-" * 50)
print(llamar_llm(prompt_minimo_actividad, max_tokens=150))
print()

RESULTADO CON PROMPT MÍNIMO:
--------------------------------------------------
Una tarjeta de crédito es como un préstamo que te permite comprar cosas ahora y pagarlas más tarde.  

Aquí te explico cómo funciona:

1. **Solicitud:** Tú solicitas una tarjeta a una entidad financiera (banco o compañía de tarjetas). Ellos evalúan tu historial crediticio y capacidad de pago para decidir si aprueban tu solicitud.

2. **Límité de crédito:** Si te aprueban, te asignan un límite de crédito, que es el máximo que puedes gastar con la tarjeta.

3. **Compras:** Puedes usar la tarjeta para comprar bienes o servicios en cualquier lugar donde se acepte.  Cuando compras, realmente estás tomando prestado dinero del banco para pagar por esos artículos. 

4



In [27]:
# TODO: Completá el prompt con los 5 componentes
# Reemplazá cada "..." con tu texto

mi_prompt = """
Rol: Experto en finanzas personales con experiencia en educación para adultos.
Contexto: Explicado para amaas de casa sin experiencia previa en finanzas.
Instrucción: Explica cómo funciona una tarjeta de crédito.
Formato: Actores involucrados, proceso paso a paso, y una analogía final.
Output esperado: una explicación clara y sencilla que cubra los puntos anteriores.
"""

print("RESULTADO CON TU PROMPT:")
print("-" * 50)
print(llamar_llm(mi_prompt, max_tokens=300))

RESULTADO CON TU PROMPT:
--------------------------------------------------
¡Hola mamás! Hoy vamos a hablar sobre las tarjetas de crédito, que pueden ser herramientas útiles para administrar el dinero si se usan con responsabilidad. 

**¿Quiénes están involucrados?**

* **Usted:**  La persona que usa la tarjeta y realiza compras.
* **Banco emisor:** La institución financiera que le otorga la tarjeta y le presta dinero.
* **Comercio:** Las tiendas o negocios donde usted realiza las compras con su tarjeta.

**¿Cómo funciona paso a paso?**

1. **Solicitud:** Para obtener una tarjeta, primero debes solicitarla al banco. Les darás información sobre tus ingresos y gastos para que ellos decidan si te la otorgan.
2. **Límit de crédito:** El banco le asignará un límite de crédito, que es el máximo dinero que puede gastar con su tarjeta. Piénsalo como una "caja" donde puedes meter dinero prestado hasta llegar a un tope específico.
3. **Compra:** Cuando compras algo con tu tarjeta, estás utilizan